In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ray
import time
import os
from itertools import product
from aircraft_profiles import AIRCRAFT_PROFILES, get_profile, list_aircraft

# Display available aircraft
print("=" * 60)
print("AVAILABLE AIRCRAFT FOR SIMULATION")
print("=" * 60)
for ac in list_aircraft():
    profile = get_profile(ac["id"])
    print(f"\n  {ac['name']} ({ac['id']})")
    print(f"    Mass: {profile['mass']:,} kg | Wingspan: {profile['wingspan']}m")
    print(f"    Max Speed: {profile['max_speed']} km/h | Drag Cd: {profile['drag_coeff']}")
    print(f"    Wind Sensitivity: {profile['wind_sensitivity']}")
print("\n" + "=" * 60)

## Summary

### What we produced:
1. **Wind physics model** — Navier-Stokes inspired force computation with Dryden turbulence
2. **Drone flight simulator** — Python port matching the in-app Cesium physics engine
3. **34,560 simulations** — distributed across the Ray cluster
4. **Training datasets** — one `.parquet` file per aircraft type with state + optimal corrections

### Key findings:
- Heavier aircraft (X-47B) are more stable but slower to correct
- Light aircraft (Cessna) drift dramatically in crosswinds
- Turbulence adds randomness that requires adaptive (ML) correction, not fixed rules
- Crosswinds cause ~3-5x more drift than headwinds of the same speed

### Next step:
→ Open `02_training.ipynb` to train ML correction models on this data

In [ ]:
# ============================================================
# PLOT 2: Heatmap — Wind Direction × Speed → Drift (per aircraft)
# ============================================================
fig, axes = plt.subplots(1, len(AIRCRAFT_IDS), figsize=(18, 5))
fig.patch.set_facecolor('#0a1628')
fig.suptitle('LATERAL DRIFT HEATMAP BY WIND CONDITIONS', 
             color='#00e5ff', fontsize=14, fontfamily='monospace', y=1.02)

for idx, (aircraft_id, df) in enumerate(all_results.items()):
    ax = axes[idx] if len(AIRCRAFT_IDS) > 1 else axes
    ax.set_facecolor('#0d1f3c')
    
    # Pivot: wind_dir × wind_speed → mean lateral drift
    pivot = df.groupby(['wind_dir', 'wind_speed'])['lateral_drift'].max().reset_index()
    pivot_table = pivot.pivot(index='wind_dir', columns='wind_speed', values='lateral_drift')
    
    profile = get_profile(aircraft_id)
    sns.heatmap(pivot_table, ax=ax, cmap='YlOrRd', annot=True, fmt='.0f',
                cbar_kws={'label': 'Max Drift (m)'},
                linewidths=0.5, linecolor='#1a3a5c')
    ax.set_title(profile['name'], color=colors[aircraft_id], fontsize=11, fontfamily='monospace')
    ax.set_xlabel('Wind Speed (m/s)', color='#8ba4c4', fontsize=9)
    ax.set_ylabel('Wind Direction (°)', color='#8ba4c4', fontsize=9)
    ax.tick_params(colors='#4a6a8a', labelsize=8)

plt.tight_layout()
plt.savefig('data/drift_heatmaps.png', dpi=150, facecolor='#0a1628', bbox_inches='tight')
plt.show()

print("\n→ Crosswinds (90°/270°) cause the most drift. Headwinds (0°/180°) cause speed changes instead.")

In [ ]:
# ============================================================
# LOAD RESULTS (if already generated)
# ============================================================
all_results = {}
for aid in AIRCRAFT_IDS:
    path = f"data/simulation_results_{aid}.parquet"
    if os.path.exists(path):
        all_results[aid] = pd.read_parquet(path)
        print(f"Loaded {aid}: {len(all_results[aid]):,} rows")

# ============================================================
# PLOT 1: Wind Speed vs Lateral Drift — All Aircraft
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
fig.patch.set_facecolor('#0a1628')
ax.set_facecolor('#0d1f3c')

colors = {'x47b': '#00e5ff', 'cessna172': '#ffc107', 'mq9_reaper': '#00e676'}

for aircraft_id, df in all_results.items():
    # Get max lateral drift per simulation (at t=60s)
    final_states = df.groupby(['wind_speed', 'seed']).agg(
        max_drift=('lateral_drift', 'max')
    ).reset_index()
    
    # Average across seeds
    avg_drift = final_states.groupby('wind_speed')['max_drift'].mean()
    std_drift = final_states.groupby('wind_speed')['max_drift'].std()
    
    profile = get_profile(aircraft_id)
    ax.plot(avg_drift.index, avg_drift.values, 'o-', color=colors[aircraft_id], 
            label=profile['name'], linewidth=2, markersize=6)
    ax.fill_between(avg_drift.index, 
                     (avg_drift - std_drift).values, 
                     (avg_drift + std_drift).values,
                     alpha=0.15, color=colors[aircraft_id])

ax.set_xlabel('Wind Speed (m/s)', color='#8ba4c4', fontsize=11)
ax.set_ylabel('Max Lateral Drift (m)', color='#8ba4c4', fontsize=11)
ax.set_title('WIND SPEED vs LATERAL DRIFT — CROSS-AIRCRAFT COMPARISON', 
             color='#00e5ff', fontsize=13, fontfamily='monospace')
ax.legend(facecolor='#0d1f3c', edgecolor='#1a3a5c', labelcolor='#8ba4c4')
ax.tick_params(colors='#4a6a8a')
ax.grid(True, alpha=0.1, color='#1a3a5c')
ax.spines[:].set_color('#1a3a5c')
plt.tight_layout()
plt.savefig('data/wind_speed_vs_drift.png', dpi=150, facecolor='#0a1628')
plt.show()

print("\n→ Key insight: The Cessna 172 (1,111 kg) drifts MUCH more than the X-47B (20,215 kg)")
print("  This is why a single ML model cannot work for all aircraft.")

## Part 5: Visualization — Cross-Aircraft Comparison

These plots demonstrate **why each aircraft needs its own ML model**. The same wind conditions produce dramatically different effects depending on the aircraft's mass, drag, and aerodynamic profile.

In [ ]:
# ============================================================
# RUN ALL SIMULATIONS
# ============================================================
# This distributes 34,560 simulations across the Ray cluster.

print("🚀 LAUNCHING SIMULATION GRID")
print(f"   {total_sims:,} simulations across {len(AIRCRAFT_IDS)} aircraft types\n")

simulation_start = time.time()
all_results = run_ray_simulation(AIRCRAFT_IDS, batch_size=50)
total_elapsed = time.time() - simulation_start

print(f"\n{'='*60}")
print(f"ALL SIMULATIONS COMPLETE")
print(f"{'='*60}")
print(f"  Total time: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")
print(f"  Total sims: {total_sims:,}")
print(f"  Avg throughput: {total_sims/total_elapsed:.0f} sims/second")

# Save results to parquet files
os.makedirs("data", exist_ok=True)
for aircraft_id, df in all_results.items():
    path = f"data/simulation_results_{aircraft_id}.parquet"
    df.to_parquet(path, index=False)
    print(f"  Saved: {path} ({len(df):,} rows, {os.path.getsize(path)/1e6:.1f} MB)")

ray.shutdown()
print("\nRay cluster shut down.")

In [ ]:
# ============================================================
# INITIALIZE RAY CLUSTER
# ============================================================
# For Domino Data Lab: ray.init(address="auto")
# For local development: ray.init() uses all CPU cores

if ray.is_initialized():
    ray.shutdown()

ray.init()  # Change to ray.init(address="auto") for Domino cluster

print(f"\nRay Cluster Resources:")
print(f"  CPUs: {ray.cluster_resources().get('CPU', 'N/A')}")
print(f"  Memory: {ray.cluster_resources().get('memory', 0) / 1e9:.1f} GB")
print(f"  Nodes: {len(ray.nodes())}")

## Part 4: Launch Simulations on Ray Cluster

**⚠️ This cell connects to your Ray cluster and runs all simulations.**

If running locally, `ray.init()` will use all available CPU cores.
For a Domino Data Lab Ray cluster, use `ray.init(address="auto")` to connect to the existing cluster head.

In [ ]:
# ============================================================
# RAY REMOTE WORKER
# ============================================================
# Each Ray worker runs a batch of simulations independently.
# The @ray.remote decorator tells Ray this function can run on any node.

@ray.remote
def simulate_batch(aircraft_id: str, params_list: list, duration: float, dt: float) -> list:
    """
    Ray remote function: runs a batch of simulations on a single worker.
    
    Args:
        aircraft_id: Which aircraft to simulate
        params_list: List of (wind_dir, wind_speed, turbulence, heading, seed) tuples
        duration: Simulation length in seconds
        dt: Physics timestep
        
    Returns:
        List of DataFrames, one per simulation
    """
    results = []
    for wind_dir, wind_speed, turbulence, heading, seed in params_list:
        np.random.seed(seed)
        df = run_single_simulation(aircraft_id, wind_dir, wind_speed, 
                                    turbulence, initial_heading=heading,
                                    duration=duration, dt=dt)
        df["seed"] = seed
        results.append(df)
    return results


def run_ray_simulation(aircraft_ids: list, batch_size: int = 50) -> dict:
    """
    Launch the full simulation grid on the Ray cluster.
    
    Now includes initial headings in the grid so the model learns
    corrections for ALL heading × wind direction combinations.
    
    Args:
        aircraft_ids: List of aircraft to simulate
        batch_size: Number of sims per Ray task (tune for your cluster)
        
    Returns:
        Dict mapping aircraft_id → DataFrame of all simulation results
    """
    results = {}
    
    for aircraft_id in aircraft_ids:
        profile = get_profile(aircraft_id)
        print(f"\n{'='*60}")
        print(f"SIMULATING: {profile['name']} ({aircraft_id})")
        print(f"{'='*60}")
        
        # Build parameter grid — now includes initial heading
        all_params = []
        for wind_dir, wind_speed, turb, heading in product(
            WIND_DIRECTIONS, WIND_SPEEDS, TURBULENCE_LEVELS, INITIAL_HEADINGS
        ):
            for seed in range(NUM_SEEDS):
                all_params.append((wind_dir, wind_speed, turb, heading, 
                                   seed + hash(aircraft_id) % 10000))
        
        print(f"  Total simulations: {len(all_params):,}")
        print(f"  Grid: {len(WIND_DIRECTIONS)} wind_dirs × {len(WIND_SPEEDS)} speeds × "
              f"{len(TURBULENCE_LEVELS)} turb × {len(INITIAL_HEADINGS)} headings × {NUM_SEEDS} seeds")
        
        # Split into batches for Ray workers
        batches = [all_params[i:i+batch_size] for i in range(0, len(all_params), batch_size)]
        print(f"  Ray tasks: {len(batches)} (batch_size={batch_size})")
        
        # Submit all batches to Ray
        start_time = time.time()
        futures = [
            simulate_batch.remote(aircraft_id, batch, SIMULATION_DURATION, DT)
            for batch in batches
        ]
        
        # Collect results with progress tracking
        all_dfs = []
        completed = 0
        for future in futures:
            batch_results = ray.get(future)
            all_dfs.extend(batch_results)
            completed += 1
            if completed % 10 == 0 or completed == len(futures):
                elapsed = time.time() - start_time
                pct = completed / len(futures) * 100
                print(f"  Progress: {completed}/{len(futures)} tasks ({pct:.0f}%) - {elapsed:.1f}s")
        
        # Combine into single DataFrame
        df = pd.concat(all_dfs, ignore_index=True)
        elapsed = time.time() - start_time
        
        print(f"\n  ✓ Complete: {len(df):,} rows in {elapsed:.1f}s")
        print(f"  Throughput: {len(all_params)/elapsed:.0f} sims/second")
        
        results[aircraft_id] = df
    
    return results

print("Ray simulation functions defined. Ready to launch.")
print(f"Available Ray resources will be shown when ray.init() is called.")

In [ ]:
# ============================================================
# RAY SIMULATION CONFIGURATION
# ============================================================

# Simulation parameter grid
WIND_DIRECTIONS = [0, 45, 90, 135, 180, 225, 270, 315]     # degrees (wind FROM)
WIND_SPEEDS = [0, 5, 10, 15, 20, 30, 40, 50]                # m/s
TURBULENCE_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]         # intensity
INITIAL_HEADINGS = [0, 45, 90, 135, 180, 225, 270, 315]     # drone heading (degrees)
NUM_SEEDS = 5                                                 # random seeds per combo
SIMULATION_DURATION = 60.0                                    # seconds
DT = 0.1                                                     # timestep

# Aircraft to simulate
AIRCRAFT_IDS = ["x47b", "cessna172", "mq9_reaper"]

# Calculate totals
# Full grid: 8 wind_dirs × 8 wind_speeds × 6 turb × 8 headings × 5 seeds = 15,360 per aircraft
combos_per_aircraft = (len(WIND_DIRECTIONS) * len(WIND_SPEEDS) * 
                       len(TURBULENCE_LEVELS) * len(INITIAL_HEADINGS) * NUM_SEEDS)
total_sims = combos_per_aircraft * len(AIRCRAFT_IDS)
total_rows = total_sims * int(SIMULATION_DURATION / DT)

print(f"Simulation Grid:")
print(f"  Wind directions:   {len(WIND_DIRECTIONS)} → {WIND_DIRECTIONS}")
print(f"  Wind speeds:       {len(WIND_SPEEDS)} → {WIND_SPEEDS}")
print(f"  Turbulence levels: {len(TURBULENCE_LEVELS)} → {TURBULENCE_LEVELS}")
print(f"  Initial headings:  {len(INITIAL_HEADINGS)} → {INITIAL_HEADINGS}")
print(f"  Random seeds:      {NUM_SEEDS}")
print(f"  Aircraft types:    {len(AIRCRAFT_IDS)}")
print(f"")
print(f"  Sims per aircraft:  {combos_per_aircraft:,}")
print(f"  Total simulations:  {total_sims:,}")
print(f"  Total data rows:    {total_rows:,}")
print(f"  Duration per sim:   {SIMULATION_DURATION}s @ {DT}s timestep")
print(f"")
print(f"  → This covers ALL heading × wind angle combinations")
print(f"     so the model learns corrections for any flight direction")

## Part 3: Ray Parallel Simulation

This is where the **Ray cluster** becomes essential. We need to simulate:
- **8** wind directions × **8** wind speeds × **6** turbulence levels × **8** initial headings × **5** random seeds
- = **15,360 simulations per aircraft**
- × **3 aircraft types** = **46,080 total simulations**

We simulate across **all heading × wind angle combinations** because:
- A drone flying North into a 90° crosswind behaves differently from one flying East into the same wind
- The ML model receives absolute heading + absolute wind direction as inputs
- It needs training data covering the full rotation space to generalize

Each simulation runs 60 seconds at 0.1s timesteps = 600 data points per sim.
Total dataset: ~27.6 million rows.

In [ ]:
class DroneSimulator:
    """
    Standalone drone flight physics simulator.
    
    Mirrors the AircraftPhysics.ts from the Cesium flight simulator app.
    Adds wind force perturbations from the WindModel on top of base physics.
    
    This is the core simulation used by Ray workers to generate training data.
    Each simulation flies a straight course and records how wind causes deviations
    from the ideal (no-wind) path.
    """
    
    def __init__(self, aircraft_id: str, initial_heading_deg: float = 0, 
                 initial_speed_kmh: float = 200, initial_altitude: float = 500):
        self.profile = get_profile(aircraft_id)
        self.aircraft_id = aircraft_id
        
        # Physics config (matches AircraftPhysics.ts)
        self.turn_rate = np.radians(45)    # rad/s
        self.climb_rate = 20.0              # m/s
        self.gravity = 2.0                  # m/s²
        self.max_roll = np.radians(45)
        
        # State
        self.speed = initial_speed_kmh / 3.6  # Convert to m/s
        self.heading = np.radians(initial_heading_deg)
        self.pitch = 0.0
        self.roll = 0.0
        self.vertical_velocity = 0.0
        
        # Position (local ENU coordinates, meters)
        self.x = 0.0  # East
        self.y = 0.0  # North
        self.z = initial_altitude  # Up
        
    def step(self, dt: float, wind_forces: dict = None) -> dict:
        """
        Advance simulation by one timestep.
        
        Args:
            dt: Time step in seconds
            wind_forces: Optional wind force dict from WindModel.compute_forces()
            
        Returns:
            Current state dictionary
        """
        # --- BASE PHYSICS (no wind) ---
        # Maintain cruise speed (simple throttle hold)
        target_speed = self.speed  # Hold current speed
        
        # Gravity pulls down
        self.vertical_velocity -= self.gravity * dt
        
        # --- APPLY WIND FORCES ---
        if wind_forces:
            mass = self.profile["mass"]
            inertia = self.profile["correction_inertia"]
            
            # Headwind → speed change (F = ma → a = F/m)
            speed_accel = wind_forces["headwind_force"] / mass
            self.speed -= speed_accel * dt  # Headwind slows us down
            
            # Crosswind → heading drift
            heading_accel = wind_forces["heading_torque"] / (mass * self.profile["length"])
            self.heading += heading_accel * dt * (1 - inertia * 0.5)
            
            # Crosswind → lateral position drift (direct push)
            lateral_accel = wind_forces["crosswind_force"] / mass
            # Apply perpendicular to heading
            self.x += lateral_accel * dt * dt * np.cos(self.heading + np.pi/2)
            self.y += lateral_accel * dt * dt * np.sin(self.heading + np.pi/2)
            
            # Vertical gust
            vert_accel = wind_forces["vertical_force"] / mass
            self.vertical_velocity += vert_accel * dt
        
        # --- UPDATE POSITION ---
        # Forward movement along heading
        forward_distance = self.speed * dt
        self.x += forward_distance * np.sin(self.heading)
        self.y += forward_distance * np.cos(self.heading)
        self.z += self.vertical_velocity * dt
        
        # Keep heading in [0, 2π]
        self.heading = self.heading % (2 * np.pi)
        
        # Clamp altitude
        self.z = max(0, self.z)
        
        return self.get_state()
    
    def get_state(self) -> dict:
        return {
            "speed": self.speed * 3.6,  # Convert back to km/h for output
            "heading": np.degrees(self.heading),
            "pitch": np.degrees(self.pitch),
            "roll": np.degrees(self.roll),
            "vertical_velocity": self.vertical_velocity,
            "x": self.x,
            "y": self.y,
            "z": self.z,
        }


def run_single_simulation(aircraft_id: str, wind_dir: float, wind_speed: float, 
                           turbulence: float, initial_heading: float = 0,
                           duration: float = 60.0, dt: float = 0.1) -> pd.DataFrame:
    """
    Run a single flight simulation with the given wind conditions.
    
    The drone flies at a given heading (constant speed) while wind pushes it around.
    We record both the actual path AND what the ideal (no-wind) path would be,
    then compute the deviations that the ML model needs to learn to correct.
    
    IMPORTANT: We simulate across different initial headings because the relative
    angle between wind and heading determines the actual forces. A 90° crosswind
    hitting a northbound drone is identical to a 180° wind hitting an eastbound drone.
    But the model needs to learn from all combinations since it receives absolute
    heading + absolute wind direction as inputs.
    
    Args:
        aircraft_id: Which aircraft to simulate
        wind_dir: Wind direction in degrees (0=North, 90=East)
        wind_speed: Wind speed in m/s
        turbulence: Turbulence intensity (0-1)
        initial_heading: Initial flight heading in degrees
        duration: Simulation length in seconds
        dt: Physics timestep
        
    Returns:
        DataFrame with timestep-by-timestep simulation data
    """
    profile = get_profile(aircraft_id)
    wind = WindModel(wind_dir, wind_speed, turbulence)
    
    # Two simulators: one with wind, one without (ideal reference)
    sim_wind = DroneSimulator(aircraft_id, initial_heading_deg=initial_heading, 
                               initial_speed_kmh=profile["cruise_speed"])
    sim_ideal = DroneSimulator(aircraft_id, initial_heading_deg=initial_heading,
                                initial_speed_kmh=profile["cruise_speed"])
    
    records = []
    timesteps = int(duration / dt)
    
    for step in range(timesteps):
        t = step * dt
        
        # Compute wind forces
        forces = wind.compute_forces(profile, sim_wind.heading, sim_wind.speed, dt)
        
        # Step both simulators
        state_wind = sim_wind.step(dt, wind_forces=forces)
        state_ideal = sim_ideal.step(dt, wind_forces=None)
        
        # Compute deviations from ideal path
        heading_error = state_wind["heading"] - state_ideal["heading"]
        # Normalize heading error to [-180, 180]
        while heading_error > 180: heading_error -= 360
        while heading_error < -180: heading_error += 360
        
        altitude_error = state_wind["z"] - state_ideal["z"]
        lateral_drift = np.sqrt(
            (state_wind["x"] - state_ideal["x"])**2 + 
            (state_wind["y"] - state_ideal["y"])**2
        )
        
        # Compute optimal corrections (what input would bring us back to ideal)
        # These are the TARGETS for the ML model
        optimal_heading_correction = -heading_error * 0.5  # Proportional correction
        optimal_throttle_correction = (state_ideal["speed"] - state_wind["speed"]) * 0.3
        optimal_pitch_correction = -altitude_error * 0.1
        
        records.append({
            "aircraft_id": aircraft_id,
            "timestep": t,
            "initial_heading": initial_heading,
            **{f"actual_{k}": v for k, v in state_wind.items()},
            **{f"ideal_{k}": v for k, v in state_ideal.items()},
            "wind_dir": wind_dir,
            "wind_speed": wind_speed,
            "turbulence": turbulence,
            "heading_error": heading_error,
            "altitude_error": altitude_error,
            "lateral_drift": lateral_drift,
            "optimal_heading_correction": np.clip(optimal_heading_correction, -15, 15),
            "optimal_throttle_correction": np.clip(optimal_throttle_correction, -50, 50),
            "optimal_pitch_correction": np.clip(optimal_pitch_correction, -10, 10),
        })
    
    return pd.DataFrame(records)

# Quick test: single simulation
print("Running test simulation: X-47B heading East, 30 m/s wind from North...")
test_df = run_single_simulation("x47b", wind_dir=0, wind_speed=30, turbulence=0.5, 
                                 initial_heading=90, duration=10)
print(f"Generated {len(test_df)} timesteps")
print(f"\nFinal lateral drift: {test_df['lateral_drift'].iloc[-1]:.1f} m")
print(f"Max heading error: {test_df['heading_error'].abs().max():.2f}°")
print(f"Initial heading: {test_df['initial_heading'].iloc[0]}°")
test_df.head()

## Part 2: Drone Flight Simulator

Python port of the AircraftPhysics engine from the Cesium flight simulator. This standalone simulator runs without a browser — it takes aircraft state + wind forces and computes the next state, exactly mirroring the in-app physics.

**State vector**: `[speed, heading, pitch, roll, vertical_velocity, x, y, z]`
**Wind effect**: Forces from the WindModel are converted to accelerations via F=ma, then applied as perturbations to heading, speed, and altitude each timestep.

In [ ]:
class WindModel:
    """
    Simplified Navier-Stokes inspired wind force model.
    
    Computes aerodynamic forces on an aircraft given:
    - Base wind vector (direction + speed)
    - Turbulence intensity (0-1)
    - Aircraft aerodynamic profile (from GLB/specs)
    
    The turbulence uses a Dryden-approximation: Gaussian noise filtered 
    through a first-order lag to create realistic gust patterns.
    """
    
    AIR_DENSITY = 1.225  # kg/m³ at sea level
    GRAVITY = 9.81       # m/s²
    
    def __init__(self, wind_dir_deg: float, wind_speed_ms: float, turbulence: float):
        """
        Args:
            wind_dir_deg: Wind direction in degrees (0=North, 90=East)
            wind_speed_ms: Base wind speed in m/s
            turbulence: Turbulence intensity 0.0 (calm) to 1.0 (severe)
        """
        self.wind_dir_rad = np.radians(wind_dir_deg)
        self.wind_speed = wind_speed_ms
        self.turbulence = np.clip(turbulence, 0, 1)
        
        # Dryden turbulence state (filtered noise)
        self._gust_state_x = 0.0
        self._gust_state_y = 0.0
        self._gust_state_z = 0.0
        self._filter_tau = 0.5  # Time constant for gust filtering
    
    def _update_turbulence(self, dt: float) -> tuple:
        """
        Generate turbulence gusts using Dryden approximation.
        Returns (gust_x, gust_y, gust_z) in m/s.
        """
        if self.turbulence < 0.01:
            return 0.0, 0.0, 0.0
        
        # Scale noise by turbulence intensity and wind speed
        sigma = self.turbulence * max(self.wind_speed * 0.3, 2.0)
        
        # First-order filter: smooth random noise into realistic gusts
        alpha = dt / (self._filter_tau + dt)
        self._gust_state_x += alpha * (np.random.normal(0, sigma) - self._gust_state_x)
        self._gust_state_y += alpha * (np.random.normal(0, sigma) - self._gust_state_y)
        self._gust_state_z += alpha * (np.random.normal(0, sigma * 0.5) - self._gust_state_z)
        
        return self._gust_state_x, self._gust_state_y, self._gust_state_z
    
    def compute_forces(self, aircraft: dict, heading_rad: float, speed_ms: float, 
                       dt: float) -> dict:
        """
        Compute wind forces on the aircraft.
        
        Args:
            aircraft: Aircraft profile dictionary
            heading_rad: Current aircraft heading in radians
            speed_ms: Current aircraft speed in m/s
            dt: Time step in seconds
            
        Returns:
            Dictionary with force components:
            - headwind_force: Force opposing/assisting motion (N)
            - crosswind_force: Lateral force (N)
            - vertical_force: Vertical gust force (N)
            - heading_torque: Torque causing yaw (N·m)
        """
        # Decompose wind into headwind/crosswind relative to aircraft heading
        relative_wind_angle = self.wind_dir_rad - heading_rad
        
        # Headwind component (positive = into the wind)
        headwind_component = self.wind_speed * np.cos(relative_wind_angle)
        # Crosswind component (positive = from the right)
        crosswind_component = self.wind_speed * np.sin(relative_wind_angle)
        
        # Add turbulence gusts
        gust_x, gust_y, gust_z = self._update_turbulence(dt)
        headwind_component += gust_x
        crosswind_component += gust_y
        
        # --- DRAG FORCE (headwind) ---
        # F = 0.5 * rho * v² * Cd * A
        relative_airspeed = speed_ms + headwind_component
        headwind_force = (0.5 * self.AIR_DENSITY * headwind_component * abs(headwind_component) 
                         * aircraft["drag_coeff"] * aircraft["frontal_area"])
        
        # --- CROSSWIND FORCE ---
        crosswind_force = (0.5 * self.AIR_DENSITY * crosswind_component * abs(crosswind_component)
                          * aircraft["drag_coeff"] * aircraft["side_area"] 
                          * aircraft["wind_sensitivity"])
        
        # --- VERTICAL GUST ---
        vertical_force = (gust_z * aircraft["mass"] * self.GRAVITY * 0.05 
                         * aircraft["wind_sensitivity"])
        
        # --- YAW TORQUE (wind pushes tail) ---
        # Longer aircraft = more weathervaning
        heading_torque = crosswind_force * aircraft["length"] * 0.3
        
        return {
            "headwind_force": headwind_force,
            "crosswind_force": crosswind_force,
            "vertical_force": vertical_force,
            "heading_torque": heading_torque,
        }

# Quick test
wind = WindModel(wind_dir_deg=90, wind_speed_ms=20, turbulence=0.5)
profile = get_profile("x47b")
forces = wind.compute_forces(profile, heading_rad=0, speed_ms=200, dt=0.1)
print("Test wind forces on X-47B (20 m/s crosswind from East):")
for k, v in forces.items():
    print(f"  {k}: {v:.2f}")
    
print("\nSame wind on Cessna 172:")
profile_c = get_profile("cessna172")
forces_c = wind.compute_forces(profile_c, heading_rad=0, speed_ms=60, dt=0.1)
for k, v in forces_c.items():
    print(f"  {k}: {v:.2f}")
print("\n→ Notice: Cessna experiences relatively MORE force due to higher wind_sensitivity")

## Part 1: Wind Physics Model (Simplified Navier-Stokes)

### Force Equations
The wind exerts three types of forces on the aircraft:

**1. Drag Force (opposing motion)**
$$F_{drag} = \frac{1}{2} \cdot \rho \cdot v^2 \cdot C_d \cdot A_{frontal}$$

**2. Lateral (Crosswind) Force**
$$F_{lateral} = \frac{1}{2} \cdot \rho \cdot v_{crosswind}^2 \cdot C_d \cdot A_{side} \cdot sensitivity$$

**3. Vertical Gust Force**
$$F_{vertical} = turbulence \cdot \mathcal{N}(0, \sigma) \cdot mass \cdot g \cdot 0.1$$

Where:
- $\rho$ = air density (1.225 kg/m³ at sea level)
- $v$ = relative wind velocity
- $C_d$ = drag coefficient (from aircraft profile)
- $A$ = cross-sectional area (from GLB model geometry)
- $\sigma$ = turbulence intensity (0-1 scale)

### Turbulence Model
We use a **Dryden turbulence model approximation** — Gaussian noise filtered through a first-order system to create realistic gusting patterns rather than pure white noise.

# 🌬️ Drone Wind Simulation Pipeline — Ray Cluster
## Generating Training Data for ML-Based Wind Correction

### Overview
This notebook simulates thousands of drone flights under varying wind conditions using **Ray** for distributed computing. The output is a training dataset that teaches a neural network how to correct for wind disturbances.

### Why This Matters
- Different aircraft respond differently to wind (an X-47B vs a Cessna 172)
- Each aircraft needs its **own trained ML model** for wind correction
- A Ray cluster lets us run 10,000+ simulations per aircraft in minutes instead of hours
- This is the data foundation for the auto-correction system

### Pipeline
```
Aircraft Profile → Wind Physics Model → Ray Parallel Sims → Training Dataset (.parquet)
     (GLB)          (Navier-Stokes)      (11,500 per aircraft)    (state + corrections)
```